In [1]:
%config InlineBackend.figure_format='retina'

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D

from pydmd import DMD
from pydmd.bopdmd import BOPDMD
from pydmd.plotter import plot_eigs
from pydmd.plotter import plot_summary
from pydmd.preprocessing.hankel import hankel_preprocessing

/var/folders/2m/t5bb62r50jbb_r1gf5dsdy680000gr/T/ipykernel_83466/1648085735.py:5: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


# Flapping Data

Importing the marker data from bird flight to investigate the dynamic modes in flight. 



In [ ]:
# Pandas import csv
full_markers = pd.read_csv("../../data/raw/full_allBirds_unilateral_markers.csv")


# Select just one bird and one perch distance. Make sure to exclude any obstacle flights or flights with weights. 
toothless_12m = full_markers[full_markers['frameID'].str.startswith('04_12')].reset_index(drop=True)

# We will also for now just take the right data
toothless_12m = toothless_12m[(toothless_12m['IMU']==0) & (toothless_12m['Obstacle']==0) & (toothless_12m['Left']==0)].reset_index(drop=True)

# Make sure ordered by frameID
toothless_12m = toothless_12m.sort_values(by='frameID').reset_index(drop=True)

# Only keep columns that are needed for DMD
columns = ['wingtip_rot_xyz_1',	'wingtip_rot_xyz_2',	'wingtip_rot_xyz_3',	'primary_rot_xyz_1',	'primary_rot_xyz_2',	'primary_rot_xyz_3',	'secondary_rot_xyz_1',	'secondary_rot_xyz_2',	'secondary_rot_xyz_3',	'tailtip_rot_xyz_1',	'tailtip_rot_xyz_2',	'tailtip_rot_xyz_3']

markers = toothless_12m[columns].to_numpy()
time = toothless_12m['time'].to_numpy()
horzDist = toothless_12m['HorzDistance'].to_numpy()

# Make a sequence column by getting the number between 04_12_ and _ in the frame ID string
toothless_12m['Seq'] = [int(ii.split('_')[2]) for ii in toothless_12m['frameID']]


# Now to mask the data so we can look at different parts of the flight

# Initial flaps are between 12m and 9m
initial_flap = (horzDist < 12) & (horzDist > 10.6)
mid_flaps = (horzDist < 12) & (horzDist > 6.8)
shift_flaps = (horzDist < 7) & (horzDist > 4.8)
glide = (horzDist < 5.5) & (horzDist > 1.3)
landing = (horzDist < 1)


# Quick plot 2 by 2
fig, ax = plt.subplots(2, 2, figsize=(10,10))

ax[0,0].scatter(-horzDist[initial_flap], markers[initial_flap,4], s=0.1, c=time[initial_flap])
ax[0,1].scatter(-horzDist[mid_flaps], markers[mid_flaps,4], s=0.1, c=time[mid_flaps])
ax[1,0].scatter(-horzDist[glide], markers[glide,4], s=0.1, c=time[glide])
ax[1,1].scatter(-horzDist[landing], markers[landing,4], s=0.1, c=time[landing])

print(toothless_12m[initial_flap].shape)
toothless_12m[initial_flap].head()
